In [0]:
%sql

CREATE CATALOG IF NOT EXISTS medalhao_credit;

In [0]:
%sql

USE CATALOG medalhao_credit;

CREATE SCHEMA IF NOT EXISTS silver_credit;

In [0]:
%sql
USE SCHEMA silver_credit;

In [0]:
from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import col

import requests
import pandas as pd
from datetime import datetime
import time

In [0]:
catalogo = "medalhao_credit"
bronze_db_name = "bronze_credit"
silver_db_name = "silver_credit"

volume_path = "/Volumes/workspace/default/data"

print(f"Ambiente configurado: {catalogo}.{silver_db_name}")

In [0]:
# Leitura da tabela Bronze de Chamados
df_bronze = spark.table(f"{catalogo}.{bronze_db_name}.chamados")

display(df_bronze.limit(5))

In [0]:
# Célula de Diagnóstico
print("Lista exata de colunas:")
print(df_bronze.columns)

In [0]:
df_ordenado = (
    df_bronze
    .withColumnRenamed("_c0", "id_chamado") #Renomeia a coluna
    .withColumn("id_chamado", col("id_chamado").cast("int")) #Transforma tudo em int
    .filter(col("id_chamado").isNotNull())  #Remove linhas se o ID estiver vazio (lixo)
    .dropDuplicates(["id_chamado"])         #Se tiver dois IDs iguais, mantém apenas um
    .orderBy("id_chamado")
)

display(df_ordenado)

In [0]:
df_cliente_tratado = (
    df_ordenado # Continuando do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c1", "id_cliente")
    
    # 2. Tipagem SEGURA (Long em vez de Int para não quebrar CPFs)
    .withColumn("id_cliente", col("id_cliente").cast("long"))
    
    # 3. Tratamento de Nulos (Regra de Ouro)
    # Não apaga a linha (o chamado existiu), mas marca o cliente como -1 (Desconhecido)
    .fillna(-1, subset=["id_cliente"])
)

display(df_cliente_tratado)